# aw_01_G — Gates G1/G2/G3: runtime hardening, verifier freeze, eval-suite freeze

**Protocol**: §10 (gates), §7 (frozen suites). Run ONCE per protocol version.

- **G1**: end-to-end smoke (config → data → train step → eval step) on a tiny budget,
  PLUS two amendments added after Phase-2 incidents:
  - **G1-reward-transport** (2026-08-15 B6 incident): heterogeneous-family GRPO
    scenario-transport audit (x17). Arrow struct unification only manifests when
    families with different key sets are mixed — a single-family tiny smoke cannot
    catch it, so the audit runs on the real frozen prompts file.
  - **G1-env-attestation** (2026-08-16 ABI incident): vLLM compatibility probe (x18,
    dry-run — does NOT mutate the runtime). Finds the newest vLLM whose resolver
    plan leaves the image-owned ABI layer (torch/torchvision/torchaudio/triton/
    nvidia-*) untouched. The resulting exact pin is committed to
    `requirements/vllm.lock.txt`; experiment notebooks only CONSUME that pin
    (aw_09_b6 header installs vLLM only if a `vllm==` line exists).
- **G2**: verifier freeze — the verifier regression suite (expected-status fixtures:
  pass/fail/malformed/illegal/timeout/indeterminate + reward-bridge semantics) must
  pass 100%; tag the verifier version in the protocol log.
- **G3**: build and FREEZE the five eval suites (300 episodes each) and the freeze
  manifest. Commit `data/eval_suites/freeze_manifest.json`; training loaders must
  treat the eval family ids as `forbidden_family_ids` (leakage gate).

**Outputs**: freeze_manifest.json (committed), smoke run artifacts,
`runs/x17_gate_g1.json`, `runs/x18_vllm_probe.json`, vLLM pin in
`requirements/vllm.lock.txt`.

In [ ]:
# @title common header
import os

from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
os.environ["WANDB_API_KEY"] = userdata.get('WANDB_API_KEY')
os.environ["GITHUB_TOKEN"] = userdata.get('GITHUB_TOKEN')

!git clone https://{os.environ["GITHUB_TOKEN"]}@github.com/m97j/axiom-world.git
%cd axiom-world
!pip install -e . -r requirements/colab-g4.lock.txt


In [ ]:
# @title 01a_g1_smoke
!python scripts/smoke_gate_g1.py


In [ ]:
# @title 01a_g1_reward_transport_smoke — heterogeneous-family GRPO reward path (x17)
# G1 amendment (2026-08-15 B6 incident): proves the scenario transport used by
# GRPO training is Arrow-safe on the REAL frozen prompts file:
#  - legacy dict column: contamination evidence (rows mutated / None-injected)
#  - fixed scenario_json string column: 0 mismatches, 0 validation failures
#  - reward parity: oracle completions score identically with and without Arrow
# Requires the frozen prompts artifact (fetch per protocol v1.3 sha-pinning first
# if not present):
#   python scripts/fetch_dataset.py --repo m97j/axiom-playworld \
#       --file playworld_prompts.jsonl --sha256 <frozen sha> --out data/train/
!python scripts/x17_grpo_scenario_audit.py \
  --prompts data/train/playworld_prompts.jsonl \
  --out runs/x17_gate_g1.json
# verdict must be PASS (exit 0). Non-zero exit = gate FAIL — do not proceed to training.

In [ ]:
# @title 01a_g1_env_attestation — image-owned ABI baseline + vLLM compat probe (x18)
# G1 amendment (2026-08-16 ABI incident): a floating vllm range replaced
# torch 2.11.0+cu128 with 2.13.0+cu13x while torchaudio stayed cu128 →
# Qwen3ForCausalLM import failure. This probe is DRY-RUN ONLY (no runtime
# mutation): it snapshots the ABI baseline and finds the newest vLLM whose pip
# resolver plan touches NO image-owned package.
!python scripts/x18_vllm_compat_probe.py --out runs/x18_vllm_probe.json

# If a CLEAN candidate is reported, OPTIONALLY smoke-test + benchmark it in THIS
# gate runtime (mutates the env — this runtime is disposable, training is not):
#   !python scripts/x18_vllm_compat_probe.py --install --model Qwen/Qwen3-8B \
#       --out runs/x18_vllm_probe_install.json
# Then write the reported lock_line (vllm==X) into requirements/vllm.lock.txt
# and commit it together with runs/x18_vllm_probe*.json.
# If NO candidate is CLEAN: leave the lock unpinned; aw_09_b6 header will skip
# the vLLM install and training must run with
#   --override training.extra.use_vllm=false   (HF-generate fallback).

In [ ]:
# @title 01b_g2_verifier_freeze — verifier regression suite (100% required)
# G2 pass condition (protocol §10): every expected-status fixture agrees —
# pass/fail/malformed/illegal/timeout/indeterminate — across the PlayWorld
# verifiers, the general verifier, and the reward-bridge status→reward map.
# Any failure = gate FAIL: fix, re-run, and only then tag the verifier version.
!python -m pytest tests/unit/test_verifiers.py \
                  tests/unit/test_general_verifier.py \
                  tests/unit/test_reward_bridge.py -q
# After passing: Record verifier version tag in protocol log
!git rev-parse --short HEAD | xargs -I{} echo "G2 verifier freeze tag: verifier-{}"

In [ ]:
# @title 01c_g3_eval_freeze
!python scripts/build_eval_suites.py --episodes-per-suite 300
# Commit data/eval_suites/freeze_manifest.json to the repo after this cell.


## Gate checklist
- [ ] G1 smoke passed (no exceptions, artifacts written)
- [ ] G1 reward-transport audit (x17) verdict PASS on the frozen prompts file — `runs/x17_gate_g1.json` archived
- [ ] G1 env attestation (x18) run — baseline recorded; exact `vllm==` pin committed to `requirements/vllm.lock.txt` (or explicitly left unpinned → HF-generate fallback documented)
- [ ] G2 verifier regression suite 100% pass — verifier version tag recorded in the protocol log
- [ ] G3 freeze manifest committed — fingerprint recorded in the protocol log